# Broadcasting : aligner une référence par colonne

Soustraire les bonnes valeurs sans fabriquer une fausse correspondance.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#broadcasting-pas-a-pas).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous conservons le tableau pédagogique [[1,2],[3,4],[5,6]]. Sa référence par colonne est [[3,4]]. Chaque ligne doit soustraire ces deux mêmes nombres.

| Ligne source | Référence utilisée | Résultat |
| --- | --- | --- |
| [1,2] | [3,4] | [−2,−2] |
| [3,4] | [3,4] | [0,0] |
| [5,6] | [3,4] | [2,2] |

La référence a la forme (1,2). Sa seule ligne est réutilisée pour les trois extraits ; les deux colonnes restent distinctes.

## Démonstration 1 · Étendre un axe de taille un

Séparer les valeurs sources et le résultat de la soustraction.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

X est (3,2), reference est (1,2). Le dernier axe vaut deux dans les deux objets.

In [4]:
X = torch.tensor([[1., 2.], [3., 4.], [5., 6.]])
reference = torch.tensor([[3., 4.]])


In [5]:
observer('X', X)
observer('reference', reference)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]
reference forme (1, 2) type torch.float32 requires_grad False
  valeurs : [[3.0, 4.0]]


### Calculer l’état suivant

L’axe unitaire de reference est compatible avec les trois lignes. X-reference crée le tableau centré.

In [6]:
centres = X - reference


In [7]:
observer('centres', centres)

centres forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[-2.0, -2.0], [0.0, 0.0], [2.0, 2.0]]


### Calculer l’état suivant

expand_as présente les valeurs répétées logiquement comme X. Nous l’utilisons seulement pour comprendre la diffusion.

In [8]:
reference_explicitement = reference.expand_as(X)


In [9]:
observer('reference_explicitement', reference_explicitement)

reference_explicitement forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[3.0, 4.0], [3.0, 4.0], [3.0, 4.0]]


### Lire les sorties

La soustraction n’a pas de suffixe de modification en place. Les deux sources gardent leurs valeurs.

In [10]:
print("reference", reference.tolist(), tuple(reference.shape))
print("alignee", reference_explicitement.tolist(), tuple(reference_explicitement.shape))
print("centres", centres.tolist(), tuple(centres.shape))
print("sources inchangees", X.tolist(), reference.tolist())

reference [[3.0, 4.0]] (1, 2)
alignee [[3.0, 4.0], [3.0, 4.0], [3.0, 4.0]] (3, 2)
centres [[-2.0, -2.0], [0.0, 0.0], [2.0, 2.0]] (3, 2)
sources inchangees [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]] [[3.0, 4.0]]


Des tailles compatibles permettent le calcul, mais ne prouvent pas que la référence correspond au bon axe. Nommer lignes et colonnes reste nécessaire.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Calculez X−référence, avec une correction par colonne.

In [11]:
exo_x = torch.tensor([[1.,2.],[3.,4.],[5.,6.]])
exo_reference = torch.tensor([1.,2.])
exo_resultat = ...  # Remplacer les points par votre réponse.
if exo_resultat is Ellipsis or exo_resultat is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    torch.testing.assert_close(torch.as_tensor(exo_resultat),exo_x-exo_reference)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

La référence de taille 2 est réutilisée sur les trois lignes.

</details>

### Exercice 2

Cette fois, soustrayez une référence par ligne : 1, 3, puis 5. Écrivez le tableau de références.

In [12]:
exo_x = torch.tensor([[1.,2.],[3.,4.],[5.,6.]])
exo_reference = ...  # Remplacer les points par votre réponse.
if exo_reference is Ellipsis or exo_reference is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_reference.shape) == (3,1)
    torch.testing.assert_close(exo_x-exo_reference,torch.tensor([[0.,1.],[0.,1.],[0.,1.]]))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Il faut garder l’axe des colonnes avec une taille un : (3,1), et non (3,).

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.tensor([[1.,2.],[3.,4.],[5.,6.]])
exo_reference = torch.tensor([1.,2.])
exo_resultat = torch.tensor([[0.,0.],[2.,2.],[4.,4.]])
if exo_resultat is Ellipsis or exo_resultat is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    torch.testing.assert_close(torch.as_tensor(exo_resultat),exo_x-exo_reference)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.tensor([[1.,2.],[3.,4.],[5.,6.]])
exo_reference = torch.tensor([[1.],[3.],[5.]])
if exo_reference is Ellipsis or exo_reference is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_reference.shape) == (3,1)
    torch.testing.assert_close(exo_x-exo_reference,torch.tensor([[0.,1.],[0.,1.],[0.,1.]]))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [13]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()